# Explore the platform

A tour of what the lake holds and of the guarantees that the code enforces.

Every read here uses `sdp.dal`. Nothing opens a Parquet file. That is the hard
rule, and it is what makes the point-in-time guarantee enforceable instead of
aspirational.

`dal` returns a lazy `DuckDBPyRelation`. Materialise at the edge only, with
`.pl()` for a polars frame or `.fetchall()` for Python values.

Run the whole notebook, or read it from the top. Each section is independent.

In [ ]:
import datetime as dt

import polars as pl

from sdp import dal

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(60)

con = dal.con()   # The one connection that owns every relation.
print(dal.status())

---
## 1. What is published

A partition exists only after its audit passed. A missing date therefore means
"no data" and never "bad data".

In [ ]:
for name, ds in dal.DATASETS.items():
    parts = dal.partitions(ds)
    span = f"{parts[0]} to {parts[-1]}" if parts else "nothing published"
    print(f"{name:24s} key={ds.key:10s} {len(parts):>4} partitions   {span}")

In [ ]:
# gaps() lists the XNYS sessions inside a range that have no partition.
# An empty list means the range is complete.
first, last = dal.coverage(dal.DAY_AGGS)
print("day aggregates", first, "to", last)
print("interior gaps:", dal.gaps(dal.DAY_AGGS, first, last) or "none")

---
## 2. The point-in-time guarantee

Splits and dividends hold **current state**. The endpoint has no `as_of`
parameter, so it gives the belief of the vendor *right now* about all of history.
The partition key is therefore the pull date.

Three accessors exist, and their names carry the semantics:

| Function | Meaning |
|---|---|
| `snapshot(ds, as_of)` | The newest pull at or before `as_of`. **Point-in-time.** |
| `snapshot_latest(ds)` | The most recent pull. Not point-in-time. |
| `history(ds)` | Every pull, stacked. For restatement work. |

In [ ]:
pulls = dal.partitions(dal.SPLITS)
print("pulls of massive_splits:", pulls)

# A read as of a date between the two pulls returns the older one, never the newer.
between = pulls[0] + dt.timedelta(days=3)
# Do not name this one `asof`. ASOF is a reserved word in DuckDB, so a query
# that reads `from asof` is a parser error.
snap = dal.snapshot(dal.SPLITS, between)
print(f"as of {between} the snapshot has",
      con.sql("select count(*) from snap").fetchone()[0], "rows")

In [ ]:
# The same read with no as_of would be lookahead, so dal makes you say which you
# want. Both of these are explicit about not being point-in-time.
latest = dal.snapshot_latest(dal.SPLITS)
print("latest pull rows:", con.sql("select count(*) from latest").fetchone()[0])

### The guard rails

Asking for a snapshot older than the first pull raises. That is the honest
answer. The endpoint has no `as_of` parameter, so the first pull is the oldest
snapshot that can ever exist.

In [ ]:
try:
    dal.snapshot(dal.SPLITS, dt.date(2020, 1, 1))
except dal.MissingPartition as exc:
    print("MissingPartition:", exc)

In [ ]:
# The two kinds of dataset cannot be confused. Each function refuses the wrong one.
for call, label in [
    (lambda: dal.series(dal.SPLITS), "series() on a current-state dataset"),
    (lambda: dal.snapshot(dal.DAY_AGGS, dt.date(2026, 8, 21)), "snapshot() on an event stream"),
]:
    try:
        call()
    except ValueError as exc:
        print(f"{label}\n    -> {exc}\n")

---
## 3. Restatement, and why the pull date is the partition key

This is the section that pays for the whole design. Two pulls exist, 13 days
apart. Compare them.

In [ ]:
h = dal.history(dal.SPLITS)
con.sql("select pull_date, count(*) as rows, count(distinct ticker) as tickers "
        "from h group by pull_date order by pull_date").pl()

The row count moved. Now find what changed. The obvious key is the vendor `id`.

In [ ]:
old_pull, new_pull = dal.partitions(dal.SPLITS)

con.sql(f'''
    select
        (select count(*) from h a
          where a.pull_date = date '{old_pull}'
            and not exists (select 1 from h b
                            where b.pull_date = date '{new_pull}' and b.id = a.id)
        ) as ids_only_in_the_old_pull,
        (select count(*) from h b
          where b.pull_date = date '{new_pull}'
            and not exists (select 1 from h a
                            where a.pull_date = date '{old_pull}' and a.id = b.id)
        ) as ids_only_in_the_new_pull
''').pl()

That reads as though hundreds of corporate actions disappeared. They did not.

Check whether the same event is still there under a different `id`, by matching
on `(ticker, execution_date)` instead.

In [ ]:
con.sql(f'''
    with dropped as (
        select a.* from h a
        where a.pull_date = date '{old_pull}'
          and not exists (select 1 from h b
                          where b.pull_date = date '{new_pull}' and b.id = a.id)
    ),
    new_pull as (select * from h where pull_date = date '{new_pull}')
    select
        count(*) as ids_that_vanished,
        count(*) filter (
            exists (select 1 from new_pull n
                    where n.ticker = d.ticker and n.execution_date = d.execution_date)
        ) as same_event_still_present_under_a_new_id
    from dropped d
''').pl()

**Every one of them is still present under a different id.** The vendor `id` is
not stable across pulls.

Two consequences:

1. A diff on `id` overstates the churn. It reports hundreds of deletions and
   insertions where the events did not change at all.
2. The dbt SCD Type 2 snapshot must not key on `id`. It would record a deletion
   and an insertion for every event whose id churned, on every pull. Key on
   `(ticker, execution_date)` instead, or on the identifier that decision 0007
   settles on.

Now measure the change that is real.

In [ ]:
con.sql(f'''
    with a as (select * from h where pull_date = date '{old_pull}'),
         b as (select * from h where pull_date = date '{new_pull}')
    select
        (select count(*) from a join b using (ticker, execution_date)
          where a.historical_adjustment_factor
                is distinct from b.historical_adjustment_factor) as factor_restated,
        (select count(*) from b
          where not exists (select 1 from a
                            where a.ticker = b.ticker
                              and a.execution_date = b.execution_date)) as genuinely_new_events
''').pl()

A restated adjustment factor is not cosmetic. It changes every adjusted price
before that event. A backtest that read the newer snapshot to price a trade dated
before the newer pull would have used information that did not exist at the time.

The `pull_date` partitions are what make this measurable. There is no
`updated_since` field on the endpoint, so a diff of complete snapshots is the only
mechanism available.

In [ ]:
# The tickers that were restated. These are the names to look at first.
con.sql(f'''
    with a as (select * from h where pull_date = date '{old_pull}'),
         b as (select * from h where pull_date = date '{new_pull}')
    select a.ticker, a.execution_date, a.adjustment_type,
           a.historical_adjustment_factor as factor_before,
           b.historical_adjustment_factor as factor_after
    from a join b using (ticker, execution_date)
    where a.historical_adjustment_factor
          is distinct from b.historical_adjustment_factor
    order by a.execution_date desc
    limit 15
''').pl()

---
## 4. Day aggregates

Unadjusted prices, one row for each ticker and session. Unadjusted is the point.
An adjusted price restates retroactively, and that would break the immutability of
a published partition.

In [ ]:
bars = dal.day_aggs()
con.sql("select count(*) as rows, count(distinct ticker) as tickers, "
        "min(date) as first_session, max(date) as last_session from bars").pl()

In [ ]:
con.sql('''
    select date, count(*) as tickers,
           round(sum(volume * close) / 1e9, 1) as dollar_volume_bn
    from bars group by date order by date
''').pl()

A detail worth knowing before you trust a type: `volume` is a floating point
number and not an integer. Most rows are fractional, because the consolidated tape
carries fractional share quantities.

In [ ]:
con.sql('''
    select count(*) as rows,
           count(*) filter (volume <> floor(volume)) as fractional_volume,
           round(100.0 * count(*) filter (volume <> floor(volume)) / count(*), 1) as pct
    from bars
''').pl()

---
## 5. The universe, and the identifier that is still an open question

Two independent filters build the universe, and both are recomputed for each date.
The instrument filter is the simple one.

In [ ]:
names = dal.tickers_on(dal.partitions(dal.TICKERS)[-1])
con.sql('''
    select type, count(*) as n
    from names group by type order by n desc limit 12
''').pl()

In [ ]:
con.sql('''
    select
        count(*) as all_instruments,
        count(*) filter (type = 'CS') as common_stock,
        count(*) filter (type = 'CS'
                         and primary_exchange in ('XNYS','XNAS','XASE')) as after_exchange_filter
    from names
''').pl()

### The identifier problem

A ticker symbol changes, and worse, it is reused. A symbol that a delisting frees
can go to a different company. A key on ticker would join the returns of two
companies into one series, and the discontinuity looks like a fat tail.

The intent was to key on `composite_figi`. Coverage blocks it. See
`docs/decisions/0007-identifier-strategy.md`.

In [ ]:
con.sql('''
    select
        count(*) as cs_rows,
        count(*) filter (composite_figi is null) as null_composite_figi,
        count(*) filter (share_class_figi is null) as null_share_class_figi,
        count(*) filter (cik is null) as null_cik
    from names where type = 'CS'
''').pl()

In [ ]:
# The landscape is the inverse of what you would expect. FIGI is complete for
# ETFs and patchy for common stock. CIK is the other way round.
con.sql('''
    select type, count(*) as n,
           count(*) filter (composite_figi is null) as null_figi,
           count(*) filter (cik is null) as null_cik
    from names
    where type in ('CS', 'ETF', 'ADRC', 'WARRANT', 'PFD')
    group by type order by n desc
''').pl()

The number that decides this question is the exposure **after** the liquidity
filter, not before it. If the null-FIGI names are a tail of recent listings and
shells, the liquidity filter removes them and the question does not matter. That
diagnostic needs the backfill.

---
## 6. Adjustment for corporate actions

The `historical_adjustment_factor` of the vendor is **cumulative**. It already
contains every later action. The adjustment is therefore one as-of lookup and not
a chained product.

> For a price on date D, find the first event whose `execution_date` is **after**
> D. Multiply the price by the factor of that event.

Check the semantics against a known value. AAPL split 2-for-1 in 2005, 7-for-1 in
2014 and 4-for-1 in 2020.

In [ ]:
splits = dal.splits(as_of=dal.partitions(dal.SPLITS)[-1])
con.sql('''
    select execution_date, adjustment_type,
           split_from, split_to,
           split_to / split_from as ratio,
           historical_adjustment_factor
    from splits where ticker = 'AAPL' order by execution_date
''').pl()

In [ ]:
# The 2005 factor must equal 1/2 * 1/7 * 1/4 = 1/56, compounded with the later
# two splits. Reproducing it from the ratios confirms the semantics.
expected = 1 / (2 * 7 * 4)
print(f"1 / (2 * 7 * 4) = {expected:.6f}")

actual = con.sql(
    "select historical_adjustment_factor from splits "
    "where ticker = 'AAPL' and execution_date = date '2005-02-28'"
).fetchone()[0]
print(f"vendor factor for 2005-02-28 = {actual}")
print("match:", round(expected, 6) == round(actual, 6))

### The boundary is strict

Split adjustment applies overnight. On the execution date all trading is already
adjusted, including the pre-market session. The join must be `> D` and not `>= D`.
An error of one day makes one very large false return for each split and each
name.

In [ ]:
# The as-of join, written out. This is the shape that the dbt staging model needs.
con.sql('''
    with universe as (
        select ticker, date, close from bars where ticker = 'AAPL'
    )
    select u.date, u.close,
           (select s.historical_adjustment_factor
              from splits s
             where s.ticker = u.ticker
               and s.execution_date > u.date      -- strictly after
             order by s.execution_date
             limit 1) as split_factor
    from universe u
    order by u.date
    limit 10
''').pl()

A null factor means that no split follows that date, so the price needs no split
adjustment. AAPL has no split after 2020, so recent bars are already on today's
share basis.

---
## 7. The audit invariants, checked against the live data

These are the properties that the audits enforce at ingest. Confirm that they hold
on what is published.

In [ ]:
# Splits classification. Every forward_split has a ratio above 1, every
# reverse_split below 1, and every stock_dividend above 1.
con.sql('''
    select adjustment_type,
           count(*) as n,
           min(split_to / split_from) as min_ratio,
           max(split_to / split_from) as max_ratio
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# Reverse splits outnumber forward splits by about two to one. Most are
# distressed microcaps doing a 1-for-10 to keep a listing.
con.sql('''
    select adjustment_type, count(*) as n,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# RYCEF. A Rolls-Royce ADR with a 1:72 stock dividend and a factor of 0.0.
# Rolls-Royce issued C Shares instead of a cash dividend. Those shares are not
# fungible with the ordinary shares, so no valid price adjustment exists and the
# vendor emits 0.0 rather than 1/72. Treating it as a split would manufacture a
# one-day fall of 98.6%.
con.sql('''
    select ticker, execution_date, adjustment_type, split_from, split_to,
           historical_adjustment_factor
    from splits
    where historical_adjustment_factor <= 0
    order by execution_date desc
    limit 10
''').pl()

In [ ]:
# The dividend factor is different. A null there is structural and not a defect.
# It means the vendor had no price on the ex-date to compute (1 - D/P) against.
divs = dal.dividends(as_of=dal.partitions(dal.DIVIDENDS)[-1])
con.sql('''
    select count(*) as rows,
           count(*) filter (historical_adjustment_factor is null) as null_factor,
           round(100.0 * count(*) filter (historical_adjustment_factor is null)
                 / count(*), 1) as pct_null,
           count(*) filter (currency is not null and currency <> 'USD') as not_usd
    from divs
''').pl()

In [ ]:
# Restrict to the tickers that actually trade on the ingested tape. The null rate
# collapses. The nulls are dividends on securities that never trade there:
# foreign issuers, OTC names that do not report, and fund share classes.
con.sql('''
    with traded as (select distinct ticker from bars)
    select
        case when d.ticker in (select ticker from traded)
             then 'in the day aggregates' else 'not in the day aggregates' end as group_,
        count(*) as rows,
        round(100.0 * count(*) filter (d.historical_adjustment_factor is null)
              / count(*), 1) as pct_null_factor
    from divs d group by 1
''').pl()

Read that second number with care. The day aggregates presently cover a few weeks
only, so any name that delisted before this window is counted as "not traded".
That is exactly the survivorship-sensitive set. The backfill settles it.

---
## 8. What is not here yet

- **The full backfill.** The lake holds a few weeks. The daily run keeps the head
  current, so the backfill only has to reach backwards.
- **The dbt staging models.** Adjustment, universe, and the snapshot of corporate
  actions. Section 6 has the shape of the first one.
- **The identifier decision.** `docs/decisions/0007-identifier-strategy.md`. It is
  blocked on a diagnostic and not on a decision, and section 5 is that diagnostic
  at a single date.

Section 3 added one requirement that was not in the plan. The vendor `id` is not
stable across pulls, so the SCD Type 2 snapshot must key on something else.